# Predicting Crowd Density with Random Forests

This notebook demonstrates a complete mini-pipeline:

- Generate a synthetic, realistic-ish time series dataset across crowd sectors
- Engineer targets for next 15 and 30 minute densities
- Train `RandomForestRegressor` models for both horizons
- Persist trained models with `joblib`
- Evaluate with MAE and R² on a held-out test split

We keep the code self-contained so it can run in any fresh environment with Python and typical ML libraries installed.


## Setup

Install and import required libraries. If running locally and a module is missing, install with `pip install scikit-learn pandas numpy joblib matplotlib seaborn`.


In [ ]:
# Imports
import os
import math
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

# Make output directory for models
os.makedirs('ai/models', exist_ok=True)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)



## Synthetic dataset generation

We simulate multiple sectors over time at 15-minute intervals. Features mimic realistic signals:

- `current_density` increases with crowd inflow, decreases with outflow, and has noise
- `drone_count` loosely correlates with density (more drones when busy)
- `alerts_count` increases with spikes in density
- `supply_level` slowly replenishes and is consumed with density
- `weather_index` varies daily and affects density

Targets are forward-looking densities at +15m and +30m aligned by sector and time.


In [ ]:
# Generate synthetic data
num_sectors = 12
start_time = datetime(2025, 1, 1, 6, 0, 0)
num_periods = 24 * 4  # one day at 15-min intervals
freq_minutes = 15

rows = []
for sector in range(num_sectors):
    baseline = np.random.uniform(30, 120)  # base density per sector
    supply = np.random.uniform(50, 100)    # starting supply level
    for i in range(num_periods):
        ts = start_time + timedelta(minutes=freq_minutes * i)
        # Daily cycle (peaks morning and evening)
        hour = ts.hour + ts.minute/60
        cycle = (
            40 * np.sin(2 * np.pi * (hour - 8) / 24.0) +
            30 * np.sin(2 * np.pi * (hour - 18) / 24.0)
        )
        weather_index = 0.5 + 0.5 * np.sin(2 * np.pi * (i / (24 * 4))) + np.random.normal(0, 0.1)
        weather_effect = 10 * weather_index

        inflow = np.clip(np.random.normal(10 + cycle/6 + weather_effect/8, 5), 0, None)
        outflow = np.clip(np.random.normal(8 + cycle/8, 4), 0, None)
        noise = np.random.normal(0, 5)

        current_density = max(0, baseline + inflow - outflow + noise)

        # Drone policy: more drones for higher density
        drone_count = int(np.clip(np.round(current_density / 50 + np.random.normal(0, 0.5)), 0, 10))

        # Alerts spike with sudden increases
        alerts_count = int(np.clip(np.round(max(0, np.random.poisson(lam=max(0.2, (current_density - baseline) / 60)))), 0, 20))

        # Supply decreases with density, replenishes slightly
        supply = np.clip(supply - current_density * 0.02 + np.random.normal(1.0, 0.5), 0, 100)

        rows.append({
            'timestamp': ts,
            'sector_id': sector,
            'current_density': float(current_density),
            'drone_count': int(drone_count),
            'alerts_count': int(alerts_count),
            'supply_level': float(supply),
            'weather_index': float(weather_index),
        })

raw_df = pd.DataFrame(rows)
raw_df.sort_values(['sector_id', 'timestamp'], inplace=True)
raw_df.reset_index(drop=True, inplace=True)

# Create forward-looking targets per sector
raw_df['next_15min_density'] = raw_df.groupby('sector_id')['current_density'].shift(-1)
raw_df['next_30min_density'] = raw_df.groupby('sector_id')['current_density'].shift(-2)

# Drop last rows per sector without targets
df = raw_df.dropna().reset_index(drop=True)

print('Dataset shape:', df.shape)
df.head()


## Train/test split and features

We train independent models for 15 and 30-minute horizons using the same feature set:

- `sector_id` (encoded as integer)
- `current_density`, `drone_count`, `alerts_count`, `supply_level`, `weather_index`

We perform a random split to keep the example simple.


In [ ]:
feature_cols = [
    'sector_id', 'current_density', 'drone_count', 'alerts_count', 'supply_level', 'weather_index'
]

X = df[feature_cols].copy()
y15 = df['next_15min_density'].copy()
y30 = df['next_30min_density'].copy()

X_train, X_test, y15_train, y15_test, y30_train, y30_test = train_test_split(
    X, y15, y30, test_size=0.2, random_state=RANDOM_SEED
)

X_train.shape, X_test.shape


## Train RandomForest models

We fit two `RandomForestRegressor` models with moderate complexity for fast training. Hyperparameters are intentionally simple.


In [ ]:
rf15 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
rf30 = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

rf15.fit(X_train, y15_train)
rf30.fit(X_train, y30_train)

print('Models trained.')


## Save trained models

We persist the models using `joblib` to `ai/models/rf_density_15.pkl` and `ai/models/rf_density_30.pkl`.


In [ ]:
model_15_path = 'ai/models/rf_density_15.pkl'
model_30_path = 'ai/models/rf_density_30.pkl'

joblib.dump(rf15, model_15_path)
joblib.dump(rf30, model_30_path)

print('Saved:', model_15_path)
print('Saved:', model_30_path)


## Evaluate on held-out test set

We compute Mean Absolute Error (MAE) and R² for both horizons. A quick scatter plot compares predictions vs. actuals.


In [ ]:
# Predictions
pred15 = rf15.predict(X_test)
pred30 = rf30.predict(X_test)

# Metrics
mae15 = mean_absolute_error(y15_test, pred15)
r2_15 = r2_score(y15_test, pred15)
mae30 = mean_absolute_error(y30_test, pred30)
r2_30 = r2_score(y30_test, pred30)

print(f"15 min -> MAE: {mae15:.2f}, R2: {r2_15:.3f}")
print(f"30 min -> MAE: {mae30:.2f}, R2: {r2_30:.3f}")

# Quick scatter plots
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y15_test, pred15, alpha=0.5)
axes[0].set_title('15 min: Actual vs Predicted')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

axes[1].scatter(y30_test, pred30, alpha=0.5, color='orange')
axes[1].set_title('30 min: Actual vs Predicted')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')

plt.tight_layout()
plt.show()
